In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
# Load dataset — 889 aligned HIV-1 envelope sequences
# Label: 0 = VRC01 resistant, 1 = VRC01 sensitive
root_dir = os.path.dirname(os.getcwd())
df = pd.read_csv(os.path.join(root_dir, "data", "input_VRC01_IC80.csv"))
print(df.shape)
print(df['Label'].value_counts())

In [ ]:
# One-hot encode sequences into a numeric feature matrix
# Each of the 209 positions gets a binary vector over 22 possible amino acids -> 4,598 features per sequence
AA_LIST = list("ACDEFGHIKLMNPQRSTVWYO-")
aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}

def one_hot_encode(sequences):
    N, L = sequences.shape
    K = len(AA_LIST)
    X = np.zeros((N, L * K), dtype=np.float32)
    for n in range(N):
        for l in range(L):
            aa = sequences[n, l]
            if aa in aa_to_idx:
                X[n, l * K + aa_to_idx[aa]] = 1.0
    return X

sequences = np.array([list(seq) for seq in df["Sequence"]])
X = one_hot_encode(sequences)
y = df["Label"].values
print(f"Feature matrix: {X.shape}")

In [ ]:
# Nested cross-validation to find the best possible LR performance
#
# Outer loop (5 folds): evaluates generalization — each fold holds out ~178 sequences as a test set
# Inner loop (3 folds): tunes hyperparameters on the training data only, never touching the test fold
#
# Hyperparameters searched:
#   C       — regularization strength (lower = simpler model, higher = fits data more closely)
#   penalty — L1 zeroes out unimportant positions; L2 shrinks all coefficients smoothly
#
# This gives LR the best possible chance, making the comparison against the PLM as fair as possible

param_grid = {
    "C":       [0.001, 0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
}

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X, y), 1):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Standardize so all features are on the same scale before fitting
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)

    # Inner CV: try all hyperparameter combinations and pick the best by AUC
    # solver='saga' supports both L1 and L2 penalties
    base_clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42, solver="saga")
    grid_search = GridSearchCV(base_clf, param_grid, cv=inner_cv, scoring="roc_auc", n_jobs=-1)
    grid_search.fit(X_train, y_train)

    # Evaluate the best model on the held-out outer fold
    best_clf = grid_search.best_estimator_
    y_pred = best_clf.predict(X_val)
    y_prob = best_clf.predict_proba(X_val)[:, 1]

    fold_results.append({
        "Fold":         fold,
        "Best C":       grid_search.best_params_["C"],
        "Best penalty": grid_search.best_params_["penalty"],
        "Accuracy":     accuracy_score(y_val, y_pred),
        "Precision":    precision_score(y_val, y_pred, zero_division=0),
        "Recall":       recall_score(y_val, y_pred, zero_division=0),
        "F1 Score":     f1_score(y_val, y_pred, zero_division=0),
        "AUC Score":    roc_auc_score(y_val, y_prob),
    })
    print(f"Fold {fold} — best: C={grid_search.best_params_['C']}, penalty={grid_search.best_params_['penalty']}, AUC={fold_results[-1]['AUC Score']:.3f}")

results_df = pd.DataFrame(fold_results)
print(results_df)

In [ ]:
# Summary — best tuned LR vs fine-tuned PLM (5-fold CV results from benchmarking.ipynb)
# PLM was evaluated with simple 5-fold CV; LR was given nested CV with hyperparameter tuning
# Despite this advantage, if LR still falls short it strengthens the case for the PLM
metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "AUC Score"]

plm_cv = {
    "Accuracy":  (0.870, 0.021),
    "Precision": (0.833, 0.045),
    "Recall":    (0.780, 0.028),
    "F1 Score":  (0.805, 0.029),
    "AUC Score": (0.920, 0.014),
}

print(f"{'Metric':<12}  {'One-Hot LR (tuned)':>22}  {'PLM (fine-tuned CV)':>22}")
print("-" * 60)
for m in metrics:
    lr_mean, lr_std = results_df[m].mean(), results_df[m].std()
    p_mean,  p_std  = plm_cv[m]
    print(f"{m:<12}  {lr_mean:.3f} +/- {lr_std:.3f}      {p_mean:.3f} +/- {p_std:.3f}")